# Hack-A-Roo 2026, AI for Quantum track
# Notebook 4: Learned error mitigation

**Problem.** Depolarizing gate noise usually shrinks measured expectation values toward zero, though not every value and not uniformly, and other noise (thermal relaxation, for one) biases toward a different fixed point. Zero-noise extrapolation (ZNE) runs the same circuit at several artificially raised noise levels and extrapolates back to zero noise. It is the standard trick. It is unbiased only when the true response is in the polynomial family being fitted, and it amplifies shot noise, because extrapolation multiplies the measurement error by a factor larger than one. Can a regressor trained on pairs of (noisy measurement, ideal value) correct expectation values better than ZNE when every measurement is estimated from a finite number of shots?

**What this notebook already does.** It builds a depolarizing noise model, generates random circuits of varying depth, and for each circuit computes the exact ideal expectation value of several observables, the exact noisy value at three noise scales, and a shot-sampled noisy value at the same three scales. Linear and quadratic ZNE run on the sampled values as baselines. A gradient-boosting regressor sees the same three sampled values plus cheap circuit features. Every method is scored on held-out circuits, and ZNE on the exact (infinite-shot) values is reported as a reference so the shot-noise amplification is visible.

**Metric.** Mean absolute error between the mitigated and the ideal expectation value, on circuits the model never saw. Lower is better.

**Budgets, stated up front.** At inference every method sees the same three sampled values from the same shots. The unmitigated baseline is also shown with three times the shots at the base scale, which is what it could have spent instead. The learned model additionally consumed a training set of circuits with ideal labels; its size and shot cost are printed, because on hardware those labels come from somewhere.

**Hypothesis to test.** At a modest shot budget the learned model's error is below linear ZNE's, because it averages shot noise across training rows. The notebook does not assume this. It also tests only one noise model, and a model trained on one noise model has not been shown to transfer to another.

**Deliverable for the poster.** One figure (error versus circuit depth for each method) and one number per method (mean absolute error on the test circuits). Then whichever shred-out you pick at the bottom.

Run every cell top to bottom once. The base run takes about a minute. Then go to the Shred-outs section.

In [ ]:
# CONFIG CELL. Every constant, seed, and path lives here. Nothing is hardcoded downstream.

RANDOM_SEED = 42

# System
QUBIT_COUNT = 4

# Random circuits. Each circuit has a random number of layers in [1, MAX_LAYERS].
# A layer is one random single-qubit rotation on every qubit, then CX on a random subset of neighboring pairs.
NUMBER_OF_CIRCUITS = 400
MAX_LAYERS = 12
CX_PROBABILITY_PER_PAIR = 0.5

# Noise. Depolarizing error after every single-qubit and two-qubit gate.
ONE_QUBIT_DEPOLARIZING_RATE = 0.003
TWO_QUBIT_DEPOLARIZING_RATE = 0.03
NOISE_SCALE_FACTORS = [1.0, 2.0, 3.0]    # ZNE runs the circuit at these multiples of the base rates

# Shots. All the observables below are Z strings, so one set of shots in the Z basis estimates all of them at once,
# exactly as it would on hardware. Each noise scale gets its own set of shots.
SHOTS_PER_CIRCUIT = 1000

# Observables: Z on each qubit and ZZ on each neighboring pair. Built from QUBIT_COUNT below so the system size can change.
# Labels are in qiskit's order: the RIGHTMOST character is qubit 0.

# Matched-budget baseline: the unmitigated estimate with all the shots the three scales would have used, at the base scale.
MATCHED_BUDGET_MULTIPLIER = len(NOISE_SCALE_FACTORS)

# Learning
TEST_FRACTION = 0.25
GRADIENT_BOOSTING_TREES = 300
GRADIENT_BOOSTING_DEPTH = 3

# Output
OUTPUT_PATH = "learned_mitigation_dataset.pkl"
FIGURE_PATH = "learned_mitigation_figure.png"

In [ ]:
# Install. Skip this cell if your environment already has these. Tested with qiskit 2.6, qiskit-aer 0.18, scikit-learn 1.8 and 1.9.
!pip install -q "qiskit[visualization]" "qiskit-aer>=0.17" "scikit-learn>=1.5" matplotlib

In [ ]:
import time
import pickle
import numpy as np
import sklearn
import qiskit
import qiskit_aer
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, DensityMatrix, SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

random_generator = np.random.default_rng(RANDOM_SEED)
assert NOISE_SCALE_FACTORS[0] == 1.0, "the first scale factor must be the base noise; several cells index column 0 as the base scale"

OBSERVABLE_LABELS = []
for qubit_index in range(QUBIT_COUNT):
    label = ["I"] * QUBIT_COUNT
    label[QUBIT_COUNT - 1 - qubit_index] = "Z"
    OBSERVABLE_LABELS.append("".join(label))
for qubit_index in range(QUBIT_COUNT - 1):
    label = ["I"] * QUBIT_COUNT
    label[QUBIT_COUNT - 1 - qubit_index] = "Z"
    label[QUBIT_COUNT - 2 - qubit_index] = "Z"
    OBSERVABLE_LABELS.append("".join(label))
observables = [SparsePauliOp(label) for label in OBSERVABLE_LABELS]
package_versions = {"numpy": np.__version__, "qiskit": qiskit.__version__, "qiskit-aer": qiskit_aer.__version__,
                    "scikit-learn": sklearn.__version__}
print("imports done,", len(observables), "observables:", OBSERVABLE_LABELS, package_versions)

In [ ]:
# Noise models, one per scale factor.
# Depolarizing error with probability p replaces the state with the maximally mixed state with probability p.
# Scaling the rates is the simulator's shortcut for raising the noise. Real hardware cannot do this and uses
# gate folding instead. Gate folding is shred-out 4.

ONE_QUBIT_GATE_NAMES = ["rx", "ry", "rz"]
TWO_QUBIT_GATE_NAMES = ["cx"]


def build_noise_model(scale_factor):
    """Depolarizing noise model with both rates multiplied by scale_factor"""
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(
        depolarizing_error(scale_factor * ONE_QUBIT_DEPOLARIZING_RATE, 1), ONE_QUBIT_GATE_NAMES)
    noise_model.add_all_qubit_quantum_error(
        depolarizing_error(scale_factor * TWO_QUBIT_DEPOLARIZING_RATE, 2), TWO_QUBIT_GATE_NAMES)
    return noise_model


# Density-matrix simulation gives the EXACT noisy expectation value when asked for the density matrix,
# and a shot-sampled estimate when asked for counts. Both are used below.
noisy_simulators = {}
for scale_factor in NOISE_SCALE_FACTORS:
    noisy_simulators[scale_factor] = AerSimulator(method="density_matrix", noise_model=build_noise_model(scale_factor))

print("noise rates at each scale (one-qubit, two-qubit):")
for scale_factor in NOISE_SCALE_FACTORS:
    print(f"  scale {scale_factor}: {scale_factor * ONE_QUBIT_DEPOLARIZING_RATE:.4f}, {scale_factor * TWO_QUBIT_DEPOLARIZING_RATE:.4f}")

In [ ]:
# Random circuits, built gate by gate, with the features the regressor will see recorded as they are built.

FEATURE_NAMES = ["layer_count", "one_qubit_gate_count", "cx_count", "observable_weight"]


def build_random_circuit(layer_count):
    """Random rotations on every qubit then CX on random neighboring pairs, repeated layer_count times"""
    circuit = QuantumCircuit(QUBIT_COUNT)
    cx_count = 0
    for layer_index in range(layer_count):
        for qubit_index in range(QUBIT_COUNT):
            rotation_name = random_generator.choice(ONE_QUBIT_GATE_NAMES)
            rotation_angle = random_generator.uniform(0, 2 * np.pi)
            getattr(circuit, rotation_name)(rotation_angle, qubit_index)
        for qubit_index in range(QUBIT_COUNT - 1):
            if random_generator.uniform() < CX_PROBABILITY_PER_PAIR:
                circuit.cx(qubit_index, qubit_index + 1)
                cx_count += 1
    return circuit, cx_count


def exact_noisy_expectation_values(circuit, simulator):
    """Exact noisy expectation of every observable from the simulated density matrix"""
    circuit_with_save = circuit.copy()
    circuit_with_save.save_density_matrix()
    # optimization_level=0 keeps the gate names the noise model is attached to; a higher level would merge rotations
    transpiled = transpile(circuit_with_save, simulator, optimization_level=0)
    density_matrix = DensityMatrix(simulator.run(transpiled).result().data()["density_matrix"])
    return np.array([density_matrix.expectation_value(observable).real for observable in observables])


# Sign table for reading Z-string expectations off counts: observable_sign_table[observable, basis state] is +1 or -1,
# the eigenvalue of that observable on that basis state. qiskit's bitstring puts qubit 0 at the RIGHT end, and so does
# the statevector index, so int(bitstring, 2) is the basis state index.
state_indices = np.arange(2 ** QUBIT_COUNT)
observable_sign_table = np.zeros((len(OBSERVABLE_LABELS), 2 ** QUBIT_COUNT))
for observable_index, label in enumerate(OBSERVABLE_LABELS):
    parity = np.zeros(2 ** QUBIT_COUNT, dtype=int)
    for qubit_index, character in enumerate(reversed(label)):
        if character == "Z":
            parity += (state_indices >> qubit_index) & 1
    observable_sign_table[observable_index] = 1 - 2 * (parity % 2)


def sampled_expectation_values(circuit, simulator, shots):
    """Shot estimate of every observable from `shots` measurements in the Z basis"""
    circuit_with_measurement = circuit.copy()
    circuit_with_measurement.measure_all()
    transpiled = transpile(circuit_with_measurement, simulator, optimization_level=0)
    counts = simulator.run(transpiled, shots=shots,
                           seed_simulator=int(random_generator.integers(0, 2**31))).result().get_counts()
    measured_probabilities = np.zeros(2 ** QUBIT_COUNT)
    for bitstring, count in counts.items():
        measured_probabilities[int(bitstring, 2)] = count / shots
    assert abs(measured_probabilities.sum() - 1) < 1e-9
    return observable_sign_table @ measured_probabilities


def ideal_expectation_values(circuit):
    """Exact noiseless expectation of every observable from the statevector"""
    statevector = Statevector(circuit)
    return np.array([statevector.expectation_value(observable).real for observable in observables])


example_circuit, example_cx_count = build_random_circuit(3)
print(example_circuit.draw(fold=120))
base_scale = NOISE_SCALE_FACTORS[0]
top_scale = NOISE_SCALE_FACTORS[-1]
print("ideal:                        ", np.round(ideal_expectation_values(example_circuit), 4))
print(f"exact noisy, scale {base_scale}:       ", np.round(exact_noisy_expectation_values(example_circuit, noisy_simulators[base_scale]), 4))
print(f"sampled noisy, scale {base_scale}:     ", np.round(sampled_expectation_values(example_circuit, noisy_simulators[base_scale], SHOTS_PER_CIRCUIT), 4))
print(f"exact noisy, scale {top_scale}:       ", np.round(exact_noisy_expectation_values(example_circuit, noisy_simulators[top_scale]), 4))
# The sign table must agree with the exact expectation on the ideal state. Check it once.
ideal_probabilities = Statevector(example_circuit).probabilities()
assert np.allclose(observable_sign_table @ ideal_probabilities, ideal_expectation_values(example_circuit))
print("sign table agrees with the exact Pauli expectations")

In [ ]:
# Generate the dataset. One row per (circuit, observable). This is the slow cell, with progress printing.

circuits = []                 # the generated circuits themselves, saved so extensions can rerun them with other noise
row_circuit_index = []        # which circuit each row came from, so the split can be by circuit
row_features = []             # FEATURE_NAMES values
row_noisy_values = []         # shot-sampled noisy value, one entry per noise scale factor. This is what the methods see.
row_matched_budget_values = []  # shot-sampled value at the base scale with MATCHED_BUDGET_MULTIPLIER times the shots
row_exact_noisy_values = []   # exact noisy value, one entry per noise scale factor. Reference only.
row_ideal_values = []

start_time = time.time()
for circuit_index in range(NUMBER_OF_CIRCUITS):
    layer_count = int(random_generator.integers(1, MAX_LAYERS + 1))
    circuit, cx_count = build_random_circuit(layer_count)
    one_qubit_gate_count = layer_count * QUBIT_COUNT

    ideal_values = ideal_expectation_values(circuit)
    sampled_per_scale = np.array([sampled_expectation_values(circuit, noisy_simulators[scale_factor], SHOTS_PER_CIRCUIT)
                                  for scale_factor in NOISE_SCALE_FACTORS])            # (scales, observables)
    matched_budget_values = sampled_expectation_values(circuit, noisy_simulators[NOISE_SCALE_FACTORS[0]],
                                                       MATCHED_BUDGET_MULTIPLIER * SHOTS_PER_CIRCUIT)
    exact_per_scale = np.array([exact_noisy_expectation_values(circuit, noisy_simulators[scale_factor])
                                for scale_factor in NOISE_SCALE_FACTORS])              # (scales, observables)

    circuits.append(circuit)
    for observable_index, label in enumerate(OBSERVABLE_LABELS):
        observable_weight = sum(1 for character in label if character != "I")
        row_circuit_index.append(circuit_index)
        row_features.append([layer_count, one_qubit_gate_count, cx_count, observable_weight])
        row_noisy_values.append(sampled_per_scale[:, observable_index])
        row_matched_budget_values.append(matched_budget_values[observable_index])
        row_exact_noisy_values.append(exact_per_scale[:, observable_index])
        row_ideal_values.append(ideal_values[observable_index])

    if (circuit_index + 1) % 50 == 0:
        elapsed_seconds = time.time() - start_time
        remaining_seconds = elapsed_seconds / (circuit_index + 1) * (NUMBER_OF_CIRCUITS - circuit_index - 1)
        print(f"simulated {circuit_index + 1}/{NUMBER_OF_CIRCUITS} circuits  elapsed {elapsed_seconds:.0f}s  remaining about {remaining_seconds:.0f}s")

row_circuit_index = np.array(row_circuit_index)
row_features = np.array(row_features, dtype=float)
row_noisy_values = np.array(row_noisy_values)
row_matched_budget_values = np.array(row_matched_budget_values)
row_exact_noisy_values = np.array(row_exact_noisy_values)
row_ideal_values = np.array(row_ideal_values)
assert row_features.shape == (NUMBER_OF_CIRCUITS * len(OBSERVABLE_LABELS), len(FEATURE_NAMES))
assert row_noisy_values.shape == (NUMBER_OF_CIRCUITS * len(OBSERVABLE_LABELS), len(NOISE_SCALE_FACTORS))
assert row_exact_noisy_values.shape == row_noisy_values.shape
print("rows:", len(row_ideal_values))
print("mean |ideal|:", round(np.abs(row_ideal_values).mean(), 4),
      "  mean |exact noisy at scale 1|:", round(np.abs(row_exact_noisy_values[:, 0]).mean(), 4))
print("mean |sampled - exact| at the base scale (the shot noise):", round(np.abs(row_noisy_values[:, 0] - row_exact_noisy_values[:, 0]).mean(), 4))
print(f"rows where the exact noisy magnitude EXCEEDS the ideal magnitude: {(np.abs(row_exact_noisy_values[:, 0]) > np.abs(row_ideal_values)).sum()} of {len(row_ideal_values)}"
      " (depolarizing noise does not shrink every value)")

In [ ]:
# Baselines: no mitigation, linear ZNE, quadratic ZNE.
# ZNE fits a polynomial through (scale factor, noisy value) and evaluates it at scale 0.

def zne_extrapolate(noisy_values_at_scales, polynomial_degree):
    """Polynomial zero-noise extrapolation for one row, returns the value at scale 0"""
    coefficients = np.polyfit(NOISE_SCALE_FACTORS, noisy_values_at_scales, polynomial_degree)
    return np.polyval(coefficients, 0.0)


unmitigated = row_noisy_values[:, 0]
unmitigated_matched_budget = row_matched_budget_values
zne_linear = np.array([zne_extrapolate(values, 1) for values in row_noisy_values])
zne_quadratic = np.array([zne_extrapolate(values, 2) for values in row_noisy_values])
# Reference: the same extrapolations on the exact noisy values, which is what ZNE would see with infinite shots.
zne_linear_exact = np.array([zne_extrapolate(values, 1) for values in row_exact_noisy_values])
zne_quadratic_exact = np.array([zne_extrapolate(values, 2) for values in row_exact_noisy_values])
# Expectation values of Pauli observables live in [-1, 1]. Extrapolation can overshoot. Clip, as a hardware user would.
zne_linear = np.clip(zne_linear, -1, 1)
zne_quadratic = np.clip(zne_quadratic, -1, 1)
zne_linear_exact = np.clip(zne_linear_exact, -1, 1)
zne_quadratic_exact = np.clip(zne_quadratic_exact, -1, 1)

# Why ZNE amplifies shot noise: the extrapolated value is a fixed linear combination of the three measured values.
# With scales 1, 2, 3 the linear fit gives (4/3) y1 + (1/3) y2 - (2/3) y3, and the quadratic gives 3 y1 - 3 y2 + y3.
# If each y has shot-noise standard deviation s, the linear result has standard deviation about 1.5 s and the
# quadratic about 4.4 s. The bias goes down and the variance goes up.
#
# Why ZNE is biased even with exact values: a circuit with many depolarizing channels has a response in the scale
# factor that is a product of many factors (1 - c * scale), a polynomial of degree equal to the number of noisy gates.
# A quadratic fit through three points at scales 1, 2, 3 does not recover its value at 0 exactly. The exact-value ZNE
# numbers below are that model bias with no sampling error at all, and no shot budget drives the sampled ZNE below them.

print("mean absolute error on ALL rows (the learned model gets its own held-out number below):")
print(f"  unmitigated                   {np.abs(unmitigated - row_ideal_values).mean():.4f}")
print(f"  unmitigated, {MATCHED_BUDGET_MULTIPLIER}x shots         {np.abs(unmitigated_matched_budget - row_ideal_values).mean():.4f}   (same total shots as the three ZNE scales)")
print(f"  ZNE linear                    {np.abs(zne_linear - row_ideal_values).mean():.4f}")
print(f"  ZNE quadratic                 {np.abs(zne_quadratic - row_ideal_values).mean():.4f}")
print(f"  ZNE linear, exact values      {np.abs(zne_linear_exact - row_ideal_values).mean():.4f}   (infinite shots, reference)")
print(f"  ZNE quadratic, exact values   {np.abs(zne_quadratic_exact - row_ideal_values).mean():.4f}   (infinite shots, reference)")

In [ ]:
# Learned mitigator: gradient boosting from (three noisy values, circuit features) to the ideal value.
# The split is by CIRCUIT, not by row, so the test circuits are unseen in every observable.

train_circuits, test_circuits = train_test_split(
    np.arange(NUMBER_OF_CIRCUITS), test_size=TEST_FRACTION, random_state=RANDOM_SEED)
train_rows = np.isin(row_circuit_index, train_circuits)
test_rows = np.isin(row_circuit_index, test_circuits)

model_inputs = np.hstack([row_noisy_values, row_features])
MODEL_INPUT_NAMES = [f"noisy_at_scale_{scale_factor}" for scale_factor in NOISE_SCALE_FACTORS] + FEATURE_NAMES

mitigation_model = GradientBoostingRegressor(
    n_estimators=GRADIENT_BOOSTING_TREES, max_depth=GRADIENT_BOOSTING_DEPTH, random_state=RANDOM_SEED)
start_time = time.time()
mitigation_model.fit(model_inputs[train_rows], row_ideal_values[train_rows])
print(f"trained on {train_rows.sum()} rows in {time.time() - start_time:.1f}s")

learned = np.clip(mitigation_model.predict(model_inputs), -1, 1)

METHOD_NAMES = ["unmitigated", f"unmitigated, {MATCHED_BUDGET_MULTIPLIER}x shots", "ZNE linear", "ZNE quadratic", "learned",
                "ZNE linear, exact values", "ZNE quadratic, exact values"]
method_predictions = {
    "unmitigated": unmitigated,
    f"unmitigated, {MATCHED_BUDGET_MULTIPLIER}x shots": unmitigated_matched_budget,
    "ZNE linear": zne_linear,
    "ZNE quadratic": zne_quadratic,
    "learned": learned,
    "ZNE linear, exact values": zne_linear_exact,         # reference only: nobody has infinite shots
    "ZNE quadratic, exact values": zne_quadratic_exact,   # reference only: this is the model bias floor
}
test_errors = {}
for method_name in METHOD_NAMES:
    test_errors[method_name] = np.abs(method_predictions[method_name][test_rows] - row_ideal_values[test_rows])

print()
print(f"RESULTS ON {test_rows.sum()} HELD-OUT ROWS FROM {len(test_circuits)} UNSEEN CIRCUITS")
print(f"{'method':30s}{'mean abs error':>16s}{'median abs error':>18s}")
for method_name in METHOD_NAMES:
    print(f"{method_name:30s}{test_errors[method_name].mean():16.4f}{np.median(test_errors[method_name]):18.4f}")
print()
print("BUDGETS")
print(f"  inference, every sampled method:  {len(NOISE_SCALE_FACTORS)} x {SHOTS_PER_CIRCUIT} shots per circuit (the {MATCHED_BUDGET_MULTIPLIER}x unmitigated baseline spends the same total at the base scale)")
print(f"  training, learned model only:     {len(train_circuits)} circuits x {len(NOISE_SCALE_FACTORS)} scales x {SHOTS_PER_CIRCUIT} shots = {len(train_circuits) * len(NOISE_SCALE_FACTORS) * SHOTS_PER_CIRCUIT} shots,"
      f" plus an exact ideal label for each of those circuits, which hardware cannot supply")
print()
print("what the model leaned on (feature importances):")
for input_name, importance in zip(MODEL_INPUT_NAMES, mitigation_model.feature_importances_):
    print(f"  {input_name:22s} {importance:.3f}")

config = {
    "RANDOM_SEED": RANDOM_SEED,
    "QUBIT_COUNT": QUBIT_COUNT,
    "NUMBER_OF_CIRCUITS": NUMBER_OF_CIRCUITS,
    "MAX_LAYERS": MAX_LAYERS,
    "CX_PROBABILITY_PER_PAIR": CX_PROBABILITY_PER_PAIR,
    "ONE_QUBIT_DEPOLARIZING_RATE": ONE_QUBIT_DEPOLARIZING_RATE,
    "TWO_QUBIT_DEPOLARIZING_RATE": TWO_QUBIT_DEPOLARIZING_RATE,
    "NOISE_SCALE_FACTORS": NOISE_SCALE_FACTORS,
    "SHOTS_PER_CIRCUIT": SHOTS_PER_CIRCUIT,
    "MATCHED_BUDGET_MULTIPLIER": MATCHED_BUDGET_MULTIPLIER,
    "OBSERVABLE_LABELS": OBSERVABLE_LABELS,
    "package_versions": package_versions,
    "TEST_FRACTION": TEST_FRACTION,
    "GRADIENT_BOOSTING_TREES": GRADIENT_BOOSTING_TREES,
    "GRADIENT_BOOSTING_DEPTH": GRADIENT_BOOSTING_DEPTH,
    "FEATURE_NAMES": FEATURE_NAMES,
}
dataset = {
    "config": config,
    "circuits": circuits,
    "row_circuit_index": row_circuit_index,
    "row_features": row_features,
    "row_noisy_values": row_noisy_values,
    "row_matched_budget_values": row_matched_budget_values,
    "row_exact_noisy_values": row_exact_noisy_values,
    "row_ideal_values": row_ideal_values,
    "train_circuits": train_circuits,
    "test_circuits": test_circuits,
    "method_predictions": method_predictions,
}
with open(OUTPUT_PATH, "wb") as output_file:
    pickle.dump(dataset, output_file)
print("saved", OUTPUT_PATH)

In [ ]:
# Poster figure: mean absolute error versus circuit depth, one line per method, test rows only.

layer_counts_of_test_rows = row_features[test_rows, 0]
unique_layer_counts = np.unique(layer_counts_of_test_rows)

figure, axis = plt.subplots(figsize=(9, 5))
for method_name in METHOD_NAMES:
    error_by_depth = [test_errors[method_name][layer_counts_of_test_rows == layer_count].mean()
                      for layer_count in unique_layer_counts]
    if "exact values" in method_name:
        axis.plot(unique_layer_counts, error_by_depth, linestyle=":", label=method_name + " (reference)")
    else:
        axis.plot(unique_layer_counts, error_by_depth, marker="o", label=method_name)
axis.set_xlabel("circuit layers")
axis.set_ylabel("mean absolute error vs ideal")
axis.set_title(f"Error mitigation on {QUBIT_COUNT} qubits, depolarizing noise, {SHOTS_PER_CIRCUIT} shots per scale, held-out circuits")
axis.legend()
figure.tight_layout()
figure.savefig(FIGURE_PATH, dpi=200)
plt.show()
print("saved", FIGURE_PATH)

## Shred-outs

The base run is a small simulator experiment with one noise model. It does not establish that learned mitigation beats ZNE in general, and the learned model has had an advantage ZNE never gets: 300 training circuits with exact ideal labels. Pick one direction below, or invent your own. Each is written as the question your poster answers and the number that answers it. Treat each circuit as the independent unit (its seven observable rows are not independent), choose settings on a validation set of circuits, and report the test number once.

1. **Fewer scales.** ZNE needs at least two scales; linear ZNE with two is a legitimate variant, and the learned model can be trained on the base-scale value alone plus circuit features. How much does each extra scale buy each method at a fixed total shot budget? Metric: test mean absolute error versus the number of scales, at matched total shots.

2. **Generalize across noise rates.** Train at the base rates, then regenerate test circuits at half and double the rates without retraining. ZNE's bias and variance change with the rate too, so run it on the new circuits as well. Does the learned model collapse, degrade gracefully, or hold? Metric: test mean absolute error versus the noise rate it is evaluated at, both methods.

3. **Shot budget.** Sweep `SHOTS_PER_CIRCUIT` from 100 to 100000. ZNE's sampling error should fall like one over the square root of the shots down to its model-bias floor (the exact-value reference lines), and the learned model's error should flatten at its own floor. Where do they cross, and is it cheaper to give ZNE more shots or to collect training circuits for the learned model? Metric: test mean absolute error versus shots, every method on one plot, with the exact-value references as horizontal lines.

4. **Gate folding instead of rate scaling.** Real hardware cannot turn a dial on the noise rate. ZNE there replaces each gate G with G G^-1 G to triple the gate count. The inverse of `rx(t)` is `rx(-t)` and the inverse of `cx` is `cx`, so folded circuits use only gate names the noise model knows. Build the folds from those primitives, transpile at optimization level 0 so the compiler does not simplify G G^-1 G back to G, and assert that the gate counts tripled. Does the learned model trained on rate-scaled data transfer to folded data? Metric: test mean absolute error, rate-scaled versus folded, both methods, on the same circuits (they are saved in the pkl).

5. **Richer noise.** Add thermal relaxation (`qiskit_aer.noise.thermal_relaxation_error` with T1, T2, and gate times) and readout error to the noise model. Does a model trained on pure depolarizing noise transfer, and does retraining on the richer noise recover? Metric: test mean absolute error under each noise model.

6. **Predict mitigability.** Some circuits cannot be rescued at any depth. Train a classifier that predicts, from circuit features alone, whether the mitigated error will exceed 0.1. Metric: classification accuracy and the depth at which most circuits become unmitigable.

7. **Scale the system.** Raise `QUBIT_COUNT` to 6 or 8. The observables are built from `QUBIT_COUNT`, so nothing else needs editing. Density-matrix simulation grows as 4^n, so the dataset shrinks or the runtime grows. Does the learned model's advantage over ZNE grow with qubit count? Metric: test mean absolute error versus qubit count for each method.

Whatever you choose, load `learned_mitigation_dataset.pkl` for the base-run circuits, rows, split, and predictions. The `config` dict inside tells you exactly which run produced it. Keep the same train/test circuits when comparing against the base run.

In [ ]:
# Your extension starts here.
